In [2]:
import numpy as np
import pandas as pd
import glob
from PIL import Image
import os

In [ ]:
def load_png_slices(png_dir, axis=2):
    """Load PNG slices and stack into a 3D volume."""
    # Get sorted list of PNG files
    png_files = sorted(glob.glob(os.path.join(png_dir, "*.png")))
    if not png_files:
        raise ValueError(f"No PNG files found in {png_dir}")

    # Load first image to get dimensions
    first_img = Image.open(png_files[0]).convert("L")  # Grayscale
    width, height = first_img.size

    # Initialize 3D array
    volume = np.zeros((len(png_files), height, width), dtype=np.float32)

    # Load and stack slices
    for i, png_file in enumerate(png_files):
        img = Image.open(png_file).convert("L")
        volume[i] = np.array(img)

    # Reorient if needed (e.g., axial to [height, width, slices])
    if axis == 2:  # Axial
        volume = volume.transpose(1, 2, 0)  # Shape: (height, width, slices)

    return volume

load_png_slices('./ntua-parkinson-dataset/')

In [ ]:
import SimpleITK as sitk
from skimage.transform import resize
from PIL import Image
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import napari

from preprocessing import helper

img_dir = "ntua-parkinson-dataset/pd-patients/Subject46/1.MRI"
df = pd.read_csv("labelled_patients.csv")

def visualise(img, title):
    viewer = napari.Viewer(title=title)
    viewer.add_image(sitk.GetArrayFromImage(img), name='3D volume')
    napari.run()

target_shape=(64,128,128)

def preprocess_volume(img_dir, target_shape=(64,128,128)):
    '''
    TODO: add comment
    '''
    img_files = helper.sort_img_files(img_dir)
    print(img_files)

    # stacking 2D images to build 3D volumes
    slices = []
    for img_path in img_files:
        # img_path = os.path.join(img_dir, img_path)
        img = Image.open(img_path).convert('L') # grayscale
        img_array = helper.normalise(img)
        img_array = resize(img_array, (target_shape[1], target_shape[2]), anti_aliasing=True)

        # check for blank slice
        if np.std(img_array) < 1e-8:
            print(f"Empty slice detected in {img_path}")
            return None # if empty, what should i do with the volume?

        slices.append(img_array)

    data = np.stack(slices, axis=0) # ensures (D, H, W); not (H, D, W); D = num_slices
    print(f"Stacked {len(slices)} in {img_dir}, shape={data.shape}")
    print(data.shape)

    # resize depth
    if data.shape[0] != target_shape[0]:
        data = resize(data, target_shape, anti_aliasing=True, preserve_range=True)
        print(f"Resized depth to {target_shape[0]}, new shape={data.shape}")

    return data

for x in df.iterrows():
    modality = x[1]['Type']
    data = preprocess_volume(x[1]['FilePath'])
    img = sitk.GetImageFromArray(data)
    visualise(img, title='1')

['ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_001.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_002.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_003.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_004.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_005.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_006.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_007.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_008.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_009.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_010.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_011.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_012.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MRI/MPRAGE_SAG_013.png', 'ntua-parkinson-dataset/pd-patients/Subject46/1.MR